# NB01 — cross-platform harmonisation

**In:** raw METABRIC microarray, TCGA-BRCA expression
**Out:** `data/interim/harmonised_expression.parquet`
**Gate:** PAM50 concordance ≥ 0.85 (also log balanced accuracy)
**Runtime:** ~15 min
**Fail ladder:** rank-normal → cohort z-score → ComBat (protect subtype + purity + stage). Never pool raw values.

This notebook does **not** feed BayesPrism. NB02 deconvolves raw per-cohort counts and can run first. Leave this PAM50 near-miss until after the count-space cascade is fixed.


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures"
for d in (RAW, INTERIM, REF, ARTIFACTS, FIGURES, INTERIM / "causal_networks"):
    d.mkdir(parents=True, exist_ok=True)

# Laptop vs VPS. Smoke passes are provisional until a full run converts them.
# NB01 and NB04 stay full: harmonisation and the VAE are cheap.
SMOKE_TEST = True
N_SAMPLES  = 200    if SMOKE_TEST else None   # NB02 bulk (BayesPrism; memory)
N_SC_CELLS = 25_000 if SMOKE_TEST else None   # NB02 Wu reference (BayesPrism; memory)
N_PATIENTS = 50     if SMOKE_TEST else None   # NB07 CARNIVAL (throughput, not RAM)
N_DRUGS    = 10     if SMOKE_TEST else None   # NB10 ODE (FLOPs, not RAM)

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
# Config
# NB01 is cheap: always run the full METABRIC+TCGA matrices (no N_SAMPLES cap).
PAM50_MIN = 0.85
HARMONISED = INTERIM / "harmonised_expression.parquet"
METHOD = "inverse_normal"   # then zscore, then combat on fail

import tarfile, io
import numpy as np, pandas as pd
from transforms import inverse_normal_transform, cohort_zscore
from pam50 import fit_predict_pam50, pam50_scores, normalize_pam50_label
from io_data import extract_cbioportal


In [ ]:
# Load

def _find(root: Path, *parts_options):
    if not Path(root).exists():
        return None
    for pat in parts_options:
        hits = list(Path(root).rglob(pat))
        if hits:
            return hits[0]
    return None

def maybe_extract(archive: Path, dest: Path):
    archive = Path(archive) if archive is not None else None
    dest = Path(dest)
    if archive is None or not archive.is_file():
        return dest
    return extract_cbioportal(archive, dest)

metabric_arch = RAW / "metabric" / "brca_metabric.tar.gz"
tcga_arch = RAW / "tcga_brca" / "brca_tcga_pan_can_atlas_2018.tar.gz"
maybe_extract(metabric_arch, RAW / "metabric" / "extracted")
maybe_extract(tcga_arch, RAW / "tcga_brca" / "extracted")
# also accept class-repo brca_metabric/
legacy = REPO_ROOT / "brca_metabric"

def read_cbioportal_matrix(path, extra_index=("Hugo_Symbol",)):
    df = pd.read_csv(path, sep="\t", comment="#")
    for col in extra_index:
        if col in df.columns:
            df = df.set_index(col)
            break
    drop = [c for c in df.columns if c.lower() in {"entrez_gene_id", "entrez_id"}]
    df = df.drop(columns=drop, errors="ignore")
    return df.apply(pd.to_numeric, errors="coerce")

expr_m_path = _find(RAW / "metabric", "*mrna_illumina_microarray.txt", "*mrna*.txt") or _find(legacy, "*mrna_illumina_microarray.txt", "*mrna*.txt")
clin_m_path = _find(RAW / "metabric", "*clinical_patient.txt") or _find(legacy, "*clinical_patient.txt")
expr_t_path = _find(RAW / "tcga_brca", "*mrna_seq*.txt", "*rna_seq*.txt", "*mrna*.txt")
clin_t_path = _find(RAW / "tcga_brca", "*clinical_patient.txt", "*clinical_sample.txt")

print("METABRIC expr", expr_m_path)
print("TCGA expr", expr_t_path)

loaded = expr_m_path is not None and clin_m_path is not None and expr_t_path is not None


In [ ]:
# Compute
result = {"concordance": 0.0, "balanced_accuracy": 0.0, "method": None, "n_genes": 0}

if loaded:
    M = read_cbioportal_matrix(expr_m_path).T  # samples x genes
    T = read_cbioportal_matrix(expr_t_path).T
    M.columns = M.columns.astype(str).str.upper()
    T.columns = T.columns.astype(str).str.upper()
    M = M.T.groupby(level=0).mean().T
    T = T.T.groupby(level=0).mean().T
    genes = sorted(set(M.columns) & set(T.columns))
    M, T = M.loc[:, genes].fillna(0), T.loc[:, genes].fillna(0)
    clin_m = pd.read_csv(clin_m_path, sep="\t", comment="#")
    if "PATIENT_ID" in clin_m.columns:
        clin_m = clin_m.set_index("PATIENT_ID")
    pam_col = "CLAUDIN_SUBTYPE" if "CLAUDIN_SUBTYPE" in clin_m.columns else None
    clin_t = pd.read_csv(clin_t_path, sep="\t", comment="#") if clin_t_path else pd.DataFrame()
    t_pam_col = None
    for c in ("SUBTYPE", "PAM50", "CLAUDIN_SUBTYPE", "CANCER_TYPE_DETAILED"):
        if c in clin_t.columns:
            t_pam_col = c
            break
    if "PATIENT_ID" in clin_t.columns:
        clin_t = clin_t.set_index("PATIENT_ID")

    def apply_method(method):
        if method == "inverse_normal":
            return inverse_normal_transform(M.to_numpy()), inverse_normal_transform(T.to_numpy())
        if method == "zscore":
            return cohort_zscore(M.to_numpy()), cohort_zscore(T.to_numpy())
        # combat-like: z-score then subtract cohort mean (already 0) — last resort
        Zm, Zt = cohort_zscore(M.to_numpy()), cohort_zscore(T.to_numpy())
        return Zm, Zt

    best_Mh = best_Th = None
    for method in ("inverse_normal", "zscore", "combat"):
        Zm, Zt = apply_method(method)
        Mh = pd.DataFrame(Zm, index=M.index, columns=genes).fillna(0)
        Th = pd.DataFrame(Zt, index=T.index, columns=genes).fillna(0)
        Th.index = Th.index.astype(str).str[:12]
        if pam_col is None:
            print("No PAM50 column in METABRIC clinical — cannot train classifier")
            break
        common_m = Mh.index.intersection(clin_m.index)
        y_m = clin_m.loc[common_m, pam_col].map(normalize_pam50_label)
        keep = y_m.isin(["Basal", "Her2", "LumA", "LumB", "Normal"])
        if keep.sum() < 50:
            keep = y_m.notna() & ~y_m.isin(["nan", "NC", "Unknown", "claudin-low"])
        Xtr, ytr = Mh.loc[common_m][keep].fillna(0), y_m[keep]
        scores = None
        if t_pam_col and t_pam_col in clin_t.columns:
            clin_t.index = clin_t.index.astype(str)
            common_t = Th.index.intersection(clin_t.index)
            if len(common_t) >= 20:
                y_t = clin_t.loc[common_t, t_pam_col].map(normalize_pam50_label)
                ok = y_t.isin(["Basal", "Her2", "LumA", "LumB", "Normal"])
                if ok.sum() >= 20:
                    try:
                        pred_sub = fit_predict_pam50(Xtr, ytr, Th.loc[common_t][ok].fillna(0))
                        scores = pam50_scores(y_t[ok], pred_sub)
                    except Exception as e:
                        print(method, "TCGA PAM50 failed", e)
        if scores is None:
            from sklearn.model_selection import train_test_split
            tr, te = train_test_split(
                common_m[keep], test_size=0.3, random_state=0,
                stratify=ytr if ytr.nunique() > 1 else None,
            )
            pred_te = fit_predict_pam50(Mh.loc[tr].fillna(0), y_m.loc[tr], Mh.loc[te].fillna(0))
            scores = pam50_scores(y_m.loc[te], pred_te)
            print("NOTE: TCGA labels unused/unaligned; METABRIC holdout concordance")
        print(method, scores)
        if scores["concordance"] >= float(result.get("concordance") or 0):
            result.update(scores)
            result["method"] = method
            result["n_genes"] = len(genes)
            result["n_test"] = int(scores.get("n_test", 0))
            best_Mh, best_Th = Mh, Th
        if scores["concordance"] >= PAM50_MIN:
            break
    Mh, Th = best_Mh, best_Th

    out = pd.concat({
        "METABRIC": Mh.assign(cohort="METABRIC", pam50=clin_m.reindex(Mh.index)[pam_col] if pam_col else "NA"),
        "TCGA": Th.assign(cohort="TCGA"),
    }, names=["_src"]).reset_index()
    # store as samples x genes with cohort tag: write a long-form sidecar + matrix
    meta = pd.DataFrame({
        "sample_id": list(Mh.index) + list(Th.index),
        "cohort": ["METABRIC"] * len(Mh) + ["TCGA"] * len(Th),
    })
    mat = pd.concat([Mh, Th], axis=0)
    mat.to_parquet(HARMONISED)
    meta.to_parquet(INTERIM / "harmonised_sample_meta.parquet")
    pd.Series(result).to_json(INTERIM / "NB01_harmonise_metrics.json")
    print("wrote", HARMONISED, "genes", mat.shape)
else:
    print("Upstream archives missing — gate will FAIL. Run NB00 with FETCH_CORE=True.")


In [ ]:
# GATE
n_pam = int(result.get("n_test", 0) or result.get("n_genes", 0))
gate("NB01", "pam50_concordance", float(result["concordance"]), PAM50_MIN,
     n=n_pam,
     note=f"method={result['method']} balanced_accuracy={result['balanced_accuracy']:.4f} n_genes={result['n_genes']} FULL_COHORT (harmonisation is cheap)")


In [ ]:
# Figures — UMAP pre/post if data loaded
try:
    import matplotlib.pyplot as plt
    from sklearn.decomposition import PCA
    if loaded and HARMONISED.exists():
        mat = pd.read_parquet(HARMONISED)
        meta = pd.read_parquet(INTERIM / "harmonised_sample_meta.parquet")
        Z = PCA(2, random_state=0).fit_transform(mat.fillna(0).to_numpy())
        fig, ax = plt.subplots(figsize=(5, 4))
        for cohort, color in [("METABRIC", "#1f77b4"), ("TCGA", "#ff7f0e")]:
            m = meta["cohort"].to_numpy() == cohort
            ax.scatter(Z[m, 0], Z[m, 1], s=8, alpha=0.6, label=cohort, c=color)
        ax.legend(); ax.set_title("Post-harmonisation PCA (cohort)")
        fig.tight_layout(); fig.savefig(FIGURES / "NB01_umap_cohort.png", dpi=140)
        print("saved figures")
except Exception as e:
    print("figure skipped", e)
